# Re-run the chemCPA conditional-expert study

A thin wrapper around the scripts the study ran. It submits nothing by itself; it lists the
exact job sequence and then reads the small result tables that live in Git. The research is in
`reports/chemcpa_moe_v2/RESULTS.md`, not here.

**Order matters.** Step 7 is the only step that opens the held-out test drugs, and it runs after
the arms are frozen. Running it earlier would put test scores inside the selection procedure.

In [ ]:
ROOT = "/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver_chemcpa_moe_v2"
BRANCH = "research/chemcpa-conditional-moe-v2"
print(f"cd {ROOT}  # git checkout {BRANCH}")

## The pipeline, as submitted

```bash
cd $ROOT

# 0. the initialisation fix, re-run on the unchanged v1 table (goal section 0)
K=$(sbatch --parsable experiments/initfix_v2/checks.sbatch)              #  ~2 min, 1 GPU
sbatch --dependency=afterok:$K experiments/initfix_v2/rerun.sbatch      #  ~2 min, 1 GPU

# 1. one official chemCPA example, to see the model's inputs, outputs and training loop
sbatch experiments/chemcpa_moe_v2/chemcpa_example.sbatch 6              # ~15 min, 1 GPU
sbatch experiments/chemcpa_moe_v2/example_report.sbatch <last.ckpt>     #  ~3 min, 1 GPU

# 2. the v2 pass over the Tahoe shards: mean(log1p) truth + the fixed single-cell sample
B=$(sbatch --parsable experiments/chemcpa_moe_v2/build_v2.sbatch 64)    # ~25 min, 14 cores

# 3. the v2 condition table, the chemCPA AnnData and the drug embedding
P=$(sbatch --parsable --dependency=afterok:$B experiments/chemcpa_moe_v2/prepare_v2.sbatch)

# 4. minimum baselines and the v1 architectures, re-scored on the v2 truth
sbatch --dependency=afterok:$P experiments/chemcpa_moe_v2/baselines_v2.sbatch

# 5. the learning-rate scan for each arm: the official rates x 0.3, 1, 3, seed 42
for A in P0 P1 P2; do sbatch --dependency=afterok:$P experiments/chemcpa_moe_v2/sweep_cpa.sbatch $A; done

# 6. the setting validation chose, at seeds 43 and 44, then selection and the P3 gate
for A in P0 P1 P2; do sbatch experiments/chemcpa_moe_v2/seeds_cpa.sbatch $A <lr_scale>; done
sbatch experiments/chemcpa_moe_v2/aggregate_cpa.sbatch
#   P3 is trained only if reports/chemcpa_moe_v2/selection.json says train_P3: true

# 7. freeze, then open the test drugs once, for every arm, through one code path
sbatch experiments/chemcpa_moe_v2/final_eval_cpa.sbatch
```

The learning-rate scale in step 6 is whatever step 5 chose on validation;
`reports/chemcpa_moe_v2/selection.json` records it. Do not substitute a value from the report
without re-running step 5.

## The initialisation fix (goal section 0)

In [ ]:
import json
import pandas as pd

R0 = "reports/initfix_v2"
runs = pd.read_csv(f"{R0}/runs.tsv", sep="\t")
display(runs)
s = json.load(open(f"{R0}/validation_summary.json"))
print("A and C start from identical shared weights:", s["shared_init_matches_across_kinds"])
for k in ["A", "C"]:
    print(k, "seed mean of scores", s[k]["seed_mean_of_scores"],
          "| score of mean prediction", s[k]["score_of_mean_prediction"])
print("C over A:", s["C_over_A_percent"])

The two aggregates are different statistics and are kept apart everywhere:
`seed_mean_of_scores` scores each seed's model and averages the scores, which is the
pre-registered primary; `score_of_mean_prediction` averages three seeds' predictions and scores
once, which is a three-model ensemble and was never pre-registered.

## What the official chemCPA example showed

In [ ]:
ex = json.load(open("reports/chemcpa_moe_v2/chemcpa_example.json"))
print(json.dumps(ex["io"], indent=1))
print(json.dumps(ex["dataset"], indent=1))
print(ex["objective"])

## The v2 data

In [ ]:
t = json.load(open("manifests/chemcpa_moe_v2/transform.json"))
print(t["truth"]); print("v1 was:", t["v1_truth"])
print({k: t[k] for k in ["n_valid_conditions", "n_valid_train", "n_valid_val", "n_valid_test",
                         "truth_rms_against_zero"]})
print(json.load(open("manifests/chemcpa_moe_v2/anndata.json")))

## The arms

In [ ]:
R = "reports/chemcpa_moe_v2"
trials = pd.read_csv(f"{R}/trials.tsv", sep="\t")
display(trials)
sel = json.load(open(f"{R}/selection.json"))
print(json.dumps(sel["selected"], indent=1))
print(json.dumps(sel["gate"], indent=1))

In [ ]:
test = json.load(open(f"{R}/test_results.json"))
rows = {a: {k: v for k, v in r.items() if not isinstance(v, dict)}
        for a, r in test["results"].items()}
display(pd.DataFrame(rows).T)
print(json.dumps(test["bootstrap"], indent=1))

## What this notebook is not

It does not select anything, fit a corrector on test truth, or produce a figure that is not
already in the reports. Changing a setting here starts a new study, which needs its own
pre-registration and its own held-out split.